# AlphaEarth / Google Satellite Embedding

This notebook extracts AlphaEarth / Google Satellite Embedding V1 values for DINO sample.

Input data:
- Imago Google Satellite Embedding V1 small-area GeoPackages, 2024.
- DINO London-wide sample points in British National Grid coordinates.



## 1. Install packages

GeoPandas / Pyogrio are used to read GeoPackage files and run the spatial join.


In [ ]:
!pip -q install geopandas pyogrio rtree pyarrow requests tqdm


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import requests
from tqdm.auto import tqdm

import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio

pd.set_option("display.max_columns", 120)


## 2. Project paths and configuration

Setting: process **2024 only** first.


In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/GEOG0105")

RAW_DIR = BASE_DIR / "Raw Data" / "AlphaEarth"
OUT_DIR = BASE_DIR / "Outputs"
TABLE_DIR = OUT_DIR / "tables"
EMB_DIR = OUT_DIR / "embeddings"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
EMB_DIR.mkdir(parents=True, exist_ok=True)

ALPHA_GPKG_PATH = RAW_DIR / "extracted-lsoa-geopackage_2024_output.gpkg"

DINO_SAMPLE_PATH = TABLE_DIR / "dino_londonwide_sample.csv"

ALPHA_OUT = EMB_DIR / "alphaearth_2024_embeddings_dino_sample.parquet"
SUMMARY_OUT = TABLE_DIR / "alphaearth_2024_dino_sample_summary.json"

## 3. Load DINO sample points
Read the DINO sample and convert the x/y coordinates into British National Grid point data.


In [ ]:
sample = pd.read_csv(DINO_SAMPLE_PATH)
sample["sample_id"] = sample["sample_id"].astype(str)

sample_gdf = gpd.GeoDataFrame(
    sample,
    geometry=gpd.points_from_xy(sample["x"], sample["y"]),
    crs="EPSG:27700"
)

display(sample_gdf.head())

## 4. Read AlphaEarth 2024 GeoPackage
Load the 2024 AlphaEarth small-area embedding polygons and identify the 64 embedding columns.

In [ ]:
alpha = gpd.read_file(ALPHA_GPKG_PATH)

if alpha.crs is None:
    alpha = alpha.set_crs("EPSG:27700")
elif alpha.crs.to_string() != "EPSG:27700":
    alpha = alpha.to_crs("EPSG:27700")

band_cols = [
    c for c in alpha.columns
    if str(c).startswith("band") and str(c).endswith("_mean")
]

assert len(band_cols) == 64, f"Expected 64 AlphaEarth bands, got {len(band_cols)}"

display(alpha.head())

## 5. Subset AlphaEarth polygons to London
Only small-area polygons near the DINO sample are retained to reduce the computational cost of spatial join.

In [ ]:
minx, miny, maxx, maxy = sample_gdf.total_bounds
buffer_m = 5000

alpha_london = alpha.cx[
    minx - buffer_m : maxx + buffer_m,
    miny - buffer_m : maxy + buffer_m
].copy()

display(alpha_london.head())

## 6. Spatial join

Match each DINO sample point to the AlphaEarth small-area polygon in which it is located.


In [ ]:
id_cols = [
    c for c in alpha_london.columns
    if any(k in str(c).lower() for k in ["lsoa", "msoa", "code", "name", "cd", "nm"])
    and c != "geometry"
]

alpha_join = alpha_london[id_cols + band_cols + ["geometry"]].copy()

joined = gpd.sjoin(
    sample_gdf,
    alpha_join,
    how="left",
    predicate="intersects"
)

joined = (
    joined
    .sort_values("sample_id")
    .drop_duplicates("sample_id", keep="first")
    .reset_index(drop=True)
)

display(joined.head())

## 7. Rename AlphaEarth embedding columns


In [ ]:
rename_map = {}

for c in band_cols:
    n = int(str(c).replace("band", "").replace("_mean", ""))
    rename_map[c] = f"alphaearth_2024_{n:03d}"

joined = joined.rename(columns=rename_map)

alphaearth_cols = list(rename_map.values())

## 8. Save AlphaEarth embeddings



In [ ]:
out = joined.drop(columns=["geometry", "index_right"], errors="ignore").copy()

meta_cols = [
    "sample_id", "task", "x", "y", "lon", "lat",
    "borough", "postcode", "postcode_clean",
    "label_regression", "label_classification",
    "property_type", "built_form", "n_certificates"
]

meta_cols = [c for c in meta_cols if c in out.columns]
id_cols = [c for c in id_cols if c in out.columns]

final_cols = meta_cols + id_cols + alphaearth_cols
final_cols = list(dict.fromkeys(final_cols))

alpha_final = out[final_cols].copy()

alpha_final.to_parquet(ALPHA_OUT, index=False)

display(alpha_final.head())

In [ ]:
summary = {
    "embedding": "AlphaEarth / Google Satellite Embedding V1",
    "year": 2024,
    "spatial_unit": "UK small-area polygon, joined to DINO sample points",
    "output_file": str(ALPHA_OUT),
    "n_rows": int(len(alpha_final)),
    "n_alphaearth_features": int(len(alphaearth_cols)),
    "missing_feature_values": int(alpha_final[alphaearth_cols].isna().sum().sum()),
    "rows_with_all_features": int(alpha_final[alphaearth_cols].notna().all(axis=1).sum()),
    "task_counts": alpha_final["task"].value_counts().to_dict()
}

with open(SUMMARY_OUT, "w") as f:
    json.dump(summary, f, indent=2)

summary